#Install Libraries

In [1]:
#download and install langchain dependencies
!pip install -qU langchain-community faiss-cpu langchain-openai

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 27.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.8/23.8 MB 44.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 88.5/88.5 kB 3.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 21.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 64.9/64.9 kB 2.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 51.0/51.0 kB 1.7 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires requests==2.32.4, but you have requests 2.33.1 which is incompatible.


In [2]:
# importing all libraries needed
import pandas as pd #for data processing

import faiss #for faiss vector database
from langchain_community.docstore.in_memory import InMemoryDocstore #for allocating vector database storage (in RAM)
from langchain_community.vectorstores import FAISS #for langchain - faiss vector database connector

from langchain_openai import OpenAIEmbeddings, ChatOpenAI #for embedding and LLM service
from uuid import uuid4 #for generate id with uuid format

from langchain_core.documents import Document #for document format that stored to vector database collection

In [3]:
import getpass #for input password interface
import os #for read and manage python directory path and environment

from google.colab import userdata

# input openai api key
if userdata.get('OPENAI_API_KEY'):
  os.environ["OPENAI_API_KEY"] = userdata.get('OPENAI_API_KEY')
else:
  os.environ["OPENAI_API_KEY"] = getpass.getpass("Enter API key for OpenAI: ")

In [4]:
#define embedding model and llm from openai provider
embeddings = OpenAIEmbeddings(model="text-embedding-3-small")
llm = ChatOpenAI(model="gpt-4o-mini")

#Read and Store Data to Vector Database

In [5]:
#directory to dataset
data_path = "/content/drive/MyDrive/JCAIEAH-003/Notes and Hands On/Modul 3/Day 10/flipkart_com-ecommerce_sample.csv"

In [6]:
#read dataset with pandas
df = pd.read_csv(data_path)
print(df.shape) #show total row and column with format (row, column)
df.head() #show first 5 record of dataset

(20000, 15)


,uniq_id,crawl_timestamp,product_url,product_name,product_category_tree,pid,retail_price,discounted_price,image,is_FK_Advantage_product,description,product_rating,overall_rating,brand,product_specifications
0,c2d766ca982eca8304150849735ffef9,2016-03-25 22:59:23 +0000,http://www.flipkart.com/alisha-solid-women-s-c...,Alisha Solid Women's Cycling Shorts,"[""Clothing >> Women's Clothing >> Lingerie, Sl...",SRTEH2FF9KEDEFGF,999.0,379.0,"[""http://img5a.flixcart.com/image/short/u/4/a/...",False,Key Features of Alisha Solid Women's Cycling S...,No rating available,No rating available,Alisha,"{""product_specification""=>[{""key""=>""Number of ..."
1,7f7036a6d550aaa89d34c77bd39a5e48,2016-03-25 22:59:23 +0000,http://www.flipkart.com/fabhomedecor-fabric-do...,FabHomeDecor Fabric Double Sofa Bed,"[""Furniture >> Living Room Furniture >> Sofa B...",SBEEH3QGU7MFYJFY,32157.0,22646.0,"[""http://img6a.flixcart.com/image/sofa-bed/j/f...",False,FabHomeDecor Fabric Double Sofa Bed (Finish Co...,No rating available,No rating available,FabHomeDecor,"{""product_specification""=>[{""key""=>""Installati..."
2,f449ec65dcbc041b6ae5e6a32717d01b,2016-03-25 22:59:23 +0000,http://www.flipkart.com/aw-bellies/p/itmeh4grg...,AW Bellies,"[""Footwear >> Women's Footwear >> Ballerinas >...",SHOEH4GRSUBJGZXE,999.0,499.0,"[""http://img5a.flixcart.com/image/shoe/7/z/z/r...",False,Key Features of AW Bellies Sandals Wedges Heel...,No rating available,No rating available,AW,"{""product_specification""=>[{""key""=>""Ideal For""..."
3,0973b37acd0c664e3de26e97e5571454,2016-03-25 22:59:23 +0000,http://www.flipkart.com/alisha-solid-women-s-c...,Alisha Solid Women's Cycling Shorts,"[""Clothing >> Women's Clothing >> Lingerie, Sl...",SRTEH2F6HUZMQ6SJ,699.0,267.0,"[""http://img5a.flixcart.com/image/short/6/2/h/...",False,Key Features of Alisha Solid Women's Cycling S...,No rating available,No rating available,Alisha,"{""product_specification""=>[{""key""=>""Number of ..."
4,bc940ea42ee6bef5ac7cea3fb5cfbee7,2016-03-25 22:59:23 +0000,http://www.flipkart.com/sicons-all-purpose-arn...,Sicons All Purpose Arnica Dog Shampoo,"[""Pet Supplies >> Grooming >> Skin & Coat Care...",PSOEH3ZYDMSYARJ5,220.0,210.0,"[""http://img5a.flixcart.com/image/pet-shampoo/...",False,Specifications of Sicons All Purpose Arnica Do...,No rating available,No rating available,Sicons,"{""product_specification""=>[{""key""=>""Pet Type"",..."


In [7]:
data = df.copy() #copy data table
data = data.iloc[:500,].reset_index(drop=True) #take row 1 to row 500 from table
data.head() #show first 5 record of dataset

,uniq_id,crawl_timestamp,product_url,product_name,product_category_tree,pid,retail_price,discounted_price,image,is_FK_Advantage_product,description,product_rating,overall_rating,brand,product_specifications
0,c2d766ca982eca8304150849735ffef9,2016-03-25 22:59:23 +0000,http://www.flipkart.com/alisha-solid-women-s-c...,Alisha Solid Women's Cycling Shorts,"[""Clothing >> Women's Clothing >> Lingerie, Sl...",SRTEH2FF9KEDEFGF,999.0,379.0,"[""http://img5a.flixcart.com/image/short/u/4/a/...",False,Key Features of Alisha Solid Women's Cycling S...,No rating available,No rating available,Alisha,"{""product_specification""=>[{""key""=>""Number of ..."
1,7f7036a6d550aaa89d34c77bd39a5e48,2016-03-25 22:59:23 +0000,http://www.flipkart.com/fabhomedecor-fabric-do...,FabHomeDecor Fabric Double Sofa Bed,"[""Furniture >> Living Room Furniture >> Sofa B...",SBEEH3QGU7MFYJFY,32157.0,22646.0,"[""http://img6a.flixcart.com/image/sofa-bed/j/f...",False,FabHomeDecor Fabric Double Sofa Bed (Finish Co...,No rating available,No rating available,FabHomeDecor,"{""product_specification""=>[{""key""=>""Installati..."
2,f449ec65dcbc041b6ae5e6a32717d01b,2016-03-25 22:59:23 +0000,http://www.flipkart.com/aw-bellies/p/itmeh4grg...,AW Bellies,"[""Footwear >> Women's Footwear >> Ballerinas >...",SHOEH4GRSUBJGZXE,999.0,499.0,"[""http://img5a.flixcart.com/image/shoe/7/z/z/r...",False,Key Features of AW Bellies Sandals Wedges Heel...,No rating available,No rating available,AW,"{""product_specification""=>[{""key""=>""Ideal For""..."
3,0973b37acd0c664e3de26e97e5571454,2016-03-25 22:59:23 +0000,http://www.flipkart.com/alisha-solid-women-s-c...,Alisha Solid Women's Cycling Shorts,"[""Clothing >> Women's Clothing >> Lingerie, Sl...",SRTEH2F6HUZMQ6SJ,699.0,267.0,"[""http://img5a.flixcart.com/image/short/6/2/h/...",False,Key Features of Alisha Solid Women's Cycling S...,No rating available,No rating available,Alisha,"{""product_specification""=>[{""key""=>""Number of ..."
4,bc940ea42ee6bef5ac7cea3fb5cfbee7,2016-03-25 22:59:23 +0000,http://www.flipkart.com/sicons-all-purpose-arn...,Sicons All Purpose Arnica Dog Shampoo,"[""Pet Supplies >> Grooming >> Skin & Coat Care...",PSOEH3ZYDMSYARJ5,220.0,210.0,"[""http://img5a.flixcart.com/image/pet-shampoo/...",False,Specifications of Sicons All Purpose Arnica Do...,No rating available,No rating available,Sicons,"{""product_specification""=>[{""key""=>""Pet Type"",..."


In [11]:
#set up index for data to be stored in vector database
index = faiss.IndexFlatL2(len(embeddings.embed_query("JC Generative AI")))

#set up vector store with faiss vector database
#this vector store will helps you to maintain vector database, such as insert new data, delete, modify, etc.
vector_store = FAISS(
    embedding_function=embeddings,
    index=index,
    docstore=InMemoryDocstore(),
    index_to_docstore_id={},
)

In [12]:
for i,e in enumerate(data.iterrows()):
  print(f"Processing data : {i}")
  unique_id = e[1]['uniq_id']
  product_name = e[1]['product_name']
  description = str(e[1]['description']) if pd.notna(e[1]['description']) else "" # Ensure description is a string and handle NaN
  document = Document(
      page_content=description,
      metadata={
          "uniq_id": str(unique_id),
          "product_name": product_name
          },
  )
  vector_store.add_documents(documents=[document], ids=[unique_id])

Processing data : 0
Processing data : 1
Processing data : 2
Processing data : 3
Processing data : 4
Processing data : 5
Processing data : 6
Processing data : 7
Processing data : 8
Processing data : 9
Processing data : 10
Processing data : 11
Processing data : 12
Processing data : 13
Processing data : 14
Processing data : 15
Processing data : 16
Processing data : 17
Processing data : 18
Processing data : 19
Processing data : 20
Processing data : 21
Processing data : 22
Processing data : 23
Processing data : 24
Processing data : 25
Processing data : 26
Processing data : 27
Processing data : 28
Processing data : 29
Processing data : 30
Processing data : 31
Processing data : 32
Processing data : 33
Processing data : 34
Processing data : 35
Processing data : 36
Processing data : 37
Processing data : 38
Processing data : 39
Processing data : 40
Processing data : 41
Processing data : 42
Processing data : 43
Processing data : 44
Processing data : 45
Processing data : 46
Processing data : 47
Pr

In [13]:
#define path for save faiss index
path_faiss = "/content/drive/MyDrive/JCAIEAH-003/Notes and Hands On/Modul 3/Day 10/faiss_index"
#save faiss index
vector_store.save_local(path_faiss)
#create new vector store with saved index
new_vector_store = FAISS.load_local(
    path_faiss, embeddings, allow_dangerous_deserialization=True
)

#Retrieve Data from Vector Database

In [14]:
#try to retrieve data from vector database with similarity_search
results = new_vector_store.similarity_search(
    "Women's Cycling Shorts",
    k=2
)
#show the result
results

[Document(id='4044c0ac52c1ee4b28777417651faf42', metadata={'uniq_id': '4044c0ac52c1ee4b28777417651faf42', 'product_name': "Alisha Solid Women's Cycling Shorts"}, page_content="Key Features of Alisha Solid Women's Cycling Shorts Cotton Lycra White, Black, Red, Black,Specifications of Alisha Solid Women's Cycling Shorts Shorts Details Number of Contents in Sales Package Pack of 4 Fabric Cotton Lycra Type Cycling Shorts General Details Pattern Solid Ideal For Women's Fabric Care Gentle Machine Wash in Lukewarm Water, Do Not Bleach Additional Details Style Code ALTGHT4P_39 In the Box 4 shorts"),
 Document(id='0973b37acd0c664e3de26e97e5571454', metadata={'uniq_id': '0973b37acd0c664e3de26e97e5571454', 'product_name': "Alisha Solid Women's Cycling Shorts"}, page_content="Key Features of Alisha Solid Women's Cycling Shorts Cotton Lycra Black, Red,Specifications of Alisha Solid Women's Cycling Shorts Shorts Details Number of Contents in Sales Package Pack of 2 Fabric Cotton Lycra Type Cycling S

In [15]:
#parsing page content and metadata for each retrieved document
for res in results:
    print(f"Result Page Content:  {res.page_content}\nResult Metadata: [{res.metadata}]")
    print("-----"*20)

Result Page Content:  Key Features of Alisha Solid Women's Cycling Shorts Cotton Lycra White, Black, Red, Black,Specifications of Alisha Solid Women's Cycling Shorts Shorts Details Number of Contents in Sales Package Pack of 4 Fabric Cotton Lycra Type Cycling Shorts General Details Pattern Solid Ideal For Women's Fabric Care Gentle Machine Wash in Lukewarm Water, Do Not Bleach Additional Details Style Code ALTGHT4P_39 In the Box 4 shorts
Result Metadata: [{'uniq_id': '4044c0ac52c1ee4b28777417651faf42', 'product_name': "Alisha Solid Women's Cycling Shorts"}]
----------------------------------------------------------------------------------------------------
Result Page Content:  Key Features of Alisha Solid Women's Cycling Shorts Cotton Lycra Black, Red,Specifications of Alisha Solid Women's Cycling Shorts Shorts Details Number of Contents in Sales Package Pack of 2 Fabric Cotton Lycra Type Cycling Shorts General Details Pattern Solid Ideal For Women's Fabric Care Gentle Machine Wash in

In [16]:
#try to retrieve data from vector database with similarity_search_with_score
#this method will give you additional information about similarity score
results = new_vector_store.similarity_search_with_score(
    "Women's Cycling Shorts",
    k=2
)
# show the result
results

[(Document(id='4044c0ac52c1ee4b28777417651faf42', metadata={'uniq_id': '4044c0ac52c1ee4b28777417651faf42', 'product_name': "Alisha Solid Women's Cycling Shorts"}, page_content="Key Features of Alisha Solid Women's Cycling Shorts Cotton Lycra White, Black, Red, Black,Specifications of Alisha Solid Women's Cycling Shorts Shorts Details Number of Contents in Sales Package Pack of 4 Fabric Cotton Lycra Type Cycling Shorts General Details Pattern Solid Ideal For Women's Fabric Care Gentle Machine Wash in Lukewarm Water, Do Not Bleach Additional Details Style Code ALTGHT4P_39 In the Box 4 shorts"),
  np.float32(0.7805227)),
 (Document(id='0973b37acd0c664e3de26e97e5571454', metadata={'uniq_id': '0973b37acd0c664e3de26e97e5571454', 'product_name': "Alisha Solid Women's Cycling Shorts"}, page_content="Key Features of Alisha Solid Women's Cycling Shorts Cotton Lycra Black, Red,Specifications of Alisha Solid Women's Cycling Shorts Shorts Details Number of Contents in Sales Package Pack of 2 Fabric

In [17]:
#parsing page content, metadata, and similarity score for each retrieved document
for res,score in results:
    print(f"Score: {score}\nResult Page Content:  {res.page_content}\nResult Metadata: [{res.metadata}]")
    print("-----"*20)

Score: 0.7805227041244507
Result Page Content:  Key Features of Alisha Solid Women's Cycling Shorts Cotton Lycra White, Black, Red, Black,Specifications of Alisha Solid Women's Cycling Shorts Shorts Details Number of Contents in Sales Package Pack of 4 Fabric Cotton Lycra Type Cycling Shorts General Details Pattern Solid Ideal For Women's Fabric Care Gentle Machine Wash in Lukewarm Water, Do Not Bleach Additional Details Style Code ALTGHT4P_39 In the Box 4 shorts
Result Metadata: [{'uniq_id': '4044c0ac52c1ee4b28777417651faf42', 'product_name': "Alisha Solid Women's Cycling Shorts"}]
----------------------------------------------------------------------------------------------------
Score: 0.7822825312614441
Result Page Content:  Key Features of Alisha Solid Women's Cycling Shorts Cotton Lycra Black, Red,Specifications of Alisha Solid Women's Cycling Shorts Shorts Details Number of Contents in Sales Package Pack of 2 Fabric Cotton Lycra Type Cycling Shorts General Details Pattern Solid 

**search_type**: Type of search to perform. Can be “similarity”, “mmr”, or “similarity_score_threshold”.

**search_kwargs**: Keyword arguments to pass to the
search function.  
Can include things like:  
* k: Amount of documents to return (Default: 4) score_threshold: Minimum relevance threshold
* fetch_k: Amount of documents to pass to MMR algorithm
(Default: 20)
* lambda_mult: Diversity of results returned by MMR;
1 for minimum diversity and 0 for maximum. (Default: 0.5)



In [18]:
#retrieve data with non-default search_type.
#basically, default search_type is similarity_score_threshold, the code below shows how to retrieve with another search_type
retriever = new_vector_store.as_retriever(search_type="mmr", search_kwargs={"k": 1})
retriever.invoke("Women's Cycling Shorts")

[Document(id='4044c0ac52c1ee4b28777417651faf42', metadata={'uniq_id': '4044c0ac52c1ee4b28777417651faf42', 'product_name': "Alisha Solid Women's Cycling Shorts"}, page_content="Key Features of Alisha Solid Women's Cycling Shorts Cotton Lycra White, Black, Red, Black,Specifications of Alisha Solid Women's Cycling Shorts Shorts Details Number of Contents in Sales Package Pack of 4 Fabric Cotton Lycra Type Cycling Shorts General Details Pattern Solid Ideal For Women's Fabric Care Gentle Machine Wash in Lukewarm Water, Do Not Bleach Additional Details Style Code ALTGHT4P_39 In the Box 4 shorts")]